In [100]:
import os
import shutil
import random
import tensorflow as tf
import keras
from tensorflow.keras.models import Sequential # type: ignore
from tensorflow.keras.layers import Conv2D,MaxPooling2D,Flatten,Dense # type: ignore
from tensorflow.keras.preprocessing.image import ImageDataGenerator # type: ignore
from keras.models import load_model
from tensorflow.keras.preprocessing import image # type: ignore
import matplotlib.pyplot as mp
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

In [101]:
BaseDIR = r"G:\Coding Languages\Artificial Intelligence\Datasets\Flower dataset\Flower Dataset"
TrainDIR = r"G:\Coding Languages\Artificial Intelligence\Datasets\Flower dataset\Train"
TestDIR = r"G:\Coding Languages\Artificial Intelligence\Datasets\Flower dataset\Test"

In [102]:
os.listdir(BaseDIR)

['daisy', 'dandelion', 'rose', 'sunflower', 'tulip']

In [103]:
for i in os.listdir(BaseDIR):
    Path = os.path.join(BaseDIR,i)
    if os.path.isdir(Path):
        for index,Fname in enumerate(os.listdir(Path),start = 1):
            OldPath = os.path.join(Path,Fname)
            ext = os.path.splitext(Fname)[1]
            NewFname = f"{index}{ext}"
            NewPath = os.path.join(Path,NewFname)
            os.rename(OldPath,NewPath)
print("All Files Renamed Successfully")

FileExistsError: [WinError 183] Cannot create a file when that file already exists: 'G:\\Coding Languages\\Artificial Intelligence\\Datasets\\Flower dataset\\Flower Dataset\\daisy\\10.jpg' -> 'G:\\Coding Languages\\Artificial Intelligence\\Datasets\\Flower dataset\\Flower Dataset\\daisy\\2.jpg'

In [104]:
for i in os.listdir(BaseDIR):
    Path = os.path.join(BaseDIR,i)
    if os.path.isdir(Path):
        image = os.listdir(Path)
        random.shuffle(image)

        TrainCount = int(len(image) * 0.7)

        Trainimages = image[:TrainCount]
        Testimages = image[TrainCount:]

        os.makedirs(os.path.join(TrainDIR, i), exist_ok = True)
        os.makedirs(os.path.join(TestDIR, i), exist_ok = True)

        for image in Trainimages:
            shutil.copy(os.path.join(Path,image),os.path.join(TrainDIR,i,image))
        
        for image in Testimages:
            shutil.copy(os.path.join(Path,image),os.path.join(TestDIR,i,image))
 
print("Dataset split complete.")

Dataset split complete.


In [105]:
ImgSize = (300,300)
BatchSize = 32
TrainDataGen = ImageDataGenerator(rescale=1/255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True)
TrainGen = TrainDataGen.flow_from_directory(TrainDIR,target_size = ImgSize, batch_size = BatchSize, class_mode = 'categorical')

TestDataGen = ImageDataGenerator(rescale = 1/255)
TestGen = TestDataGen.flow_from_directory(TestDIR,target_size = ImgSize,batch_size = BatchSize,class_mode = 'categorical')

Found 3924 images belonging to 5 classes.
Found 2203 images belonging to 5 classes.


In [106]:
TrainGen.class_indices,TrainGen.class_mode,

({'daisy': 0, 'dandelion': 1, 'rose': 2, 'sunflower': 3, 'tulip': 4},
 'categorical')

In [107]:
TrainGen.classes,TrainGen.filenames[:5],TrainGen.samples

(array([0, 0, 0, ..., 4, 4, 4], dtype=int32),
 ['daisy\\1.jpg',
  'daisy\\10.jpg',
  'daisy\\100.jpg',
  'daisy\\101.jpg',
  'daisy\\102.jpg'],
 3924)

In [108]:
NumClasses = len(os.listdir(TrainDIR))

In [109]:
NumClasses

5

In [110]:
df = keras.utils.image_dataset_from_directory(TrainDIR,image_size = (180,180),batch_size = BatchSize, label_mode = 'categorical')

Found 3924 files belonging to 5 classes.


In [122]:
def GetModel():
    Model = Sequential()
    Model.add(Conv2D(filters = 128, kernel_size = (3,3), activation = 'relu',input_shape = (300,300,3)))
    Model.add(MaxPooling2D(pool_size = (5,5)))

    Model.add(Conv2D(filters = 128, kernel_size = (3,3), activation = 'relu'))
    Model.add(MaxPooling2D(pool_size = (5,5)))

    Model.add(Conv2D(filters = 512,kernel_size = (3,3), activation = 'relu'))
    Model.add(MaxPooling2D(pool_size = (3,3), strides = 2))

    Model.add(Flatten())
    Model.add(Dense(256,activation = 'relu'))

    Model.add(Dense(NumClasses,activation = 'softmax'))

    return Model

In [112]:
Model = GetModel()

In [113]:
Model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_12 (Conv2D)              │ (None, 298, 298, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 59, 59, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 57, 57, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_13 (MaxPooling2D) │ (None, 11, 11, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 9, 9, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_14 (MaxPooling2D) │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 5)              │         1,285 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 638,405 (2.44 MB)

 Trainable params: 638,405 (2.44 MB)

 Non-trainable params: 0 (0.00 B)

In [114]:
Model.layers

[<Conv2D name=conv2d_12, built=True>,
 <MaxPooling2D name=max_pooling2d_12, built=True>,
 <Conv2D name=conv2d_13, built=True>,
 <MaxPooling2D name=max_pooling2d_13, built=True>,
 <Conv2D name=conv2d_14, built=True>,
 <MaxPooling2D name=max_pooling2d_14, built=True>,
 <Flatten name=flatten_4, built=True>,
 <Dense name=dense_8, built=True>,
 <Dense name=dense_9, built=True>]

In [115]:
weights,biases = Model.layers[0].get_weights()

In [116]:
len(biases),len(weights)

(64, 3)

In [117]:
Model.compile(optimizer = 'adam',loss = 'categorical_crossentropy',metrics = ['accuracy'])

In [118]:
TrainDataGen = TrainDataGen.flow_from_directory(TrainDIR,target_size = ImgSize,batch_size = BatchSize, class_mode = 'categorical')
TestGen = TestDataGen.flow_from_directory(TestDIR,target_size = ImgSize,batch_size = BatchSize,class_mode = 'categorical')

Found 3924 images belonging to 5 classes.
Found 2203 images belonging to 5 classes.


In [ ]:
History = Model.fit(TrainGen,epochs = 15, validation_data = TestGen)

g:\Applications\Python 3.10.11\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/15
123/123 ━━━━━━━━━━━━━━━━━━━━ 241s 2s/step - accuracy: 0.3512 - loss: 1.4166 - val_accuracy: 0.5865 - val_loss: 1.0009
Epoch 2/15
123/123 ━━━━━━━━━━━━━━━━━━━━ 195s 2s/step - accuracy: 0.5672 - loss: 1.0519 - val_accuracy: 0.6396 - val_loss: 0.8973
Epoch 3/15
123/123 ━━━━━━━━━━━━━━━━━━━━ 190s 2s/step - accuracy: 0.6331 - loss: 0.9250 - val_accuracy: 0.6564 - val_loss: 0.9195
Epoch 4/15
123/123 ━━━━━━━━━━━━━━━━━━━━ 194s 2s/step - accuracy: 0.6678 - loss: 0.8549 - val_accuracy: 0.7149 - val_loss: 0.7551
Epoch 5/15
123/123 ━━━━━━━━━━━━━━━━━━━━ 194s 2s/step - accuracy: 0.7082 - loss: 0.7478 - val_accuracy: 0.7213 - val_loss: 0.7486
Epoch 6/15
123/123 ━━━━━━━━━━━━━━━━━━━━ 186s 2s/step - accuracy: 0.7141 - loss: 0.7643 - val_accuracy: 0.7635 - val_loss: 0.6435
Epoch 7/15
123/123 ━━━━━━━━━━━━━━━━━━━━ 234s 2s/step - accuracy: 0.7287 - loss: 0.7065 - val_accuracy: 0.7549 - val_loss: 0.6516
Epoch 8/15
123/123 ━━━━━━━━━━━━━━━━━━━━ 302s 2s/step - accuracy: 0.7400 - loss: 0.6953 - val_accu

In [121]:
Model.save(r"G:\Coding Languages\Artificial Intelligence\Models\Flowers Prediction Model.h5")

AttributeError: 'History' object has no attribute 'save'

In [86]:
Model = load_model(r"G:\Coding Languages\Artificial Intelligence\Models\Flowers Prediction Model.h5")

In [87]:
TargetSize = (300,300)
img = image.load_img(r"G:\Coding Languages\Artificial Intelligence\Datasets\Flower dataset\Test\dandelion\4.jpg",target_size=TargetSize)
img = image.img_to_array(img) / 255.0
img = np.expand_dims(img,axis = 0)
Prediction = Model(img)

AttributeError: 'str' object has no attribute 'load_img'

In [ ]:
Prediction

<tf.Tensor: shape=(1, 5), dtype=float32, numpy=
array([[1.4136541e-04, 9.9955672e-01, 2.3597974e-04, 5.2904506e-05,
        1.3069507e-05]], dtype=float32)>

In [ ]:
TH = 0.5
PredictedClass = int(Prediction[0][0] > TH)
ClassIndices = TrainGen.class_indices
labels = {v:k for k,v in ClassIndices.items()}
print("The Image Is Of A:",labels[PredictedClass])

The Image Is Of A: daisy


In [ ]:
print(TrainGen.class_indices)
print(TrainGen.classes)
np.bincount(TrainGen.classes)

{'daisy': 0, 'dandelion': 1, 'rose': 2, 'sunflower': 3, 'tulip': 4}
[0 0 0 ... 4 4 4]


array([534, 736, 548, 513, 688])